In [ ]:
from sts.data.ticker import Ticker
from sts.quant.trend import get_trend_score
import pandas as pd
import plotly.io as pio
from sts.quant.candle import Candle
import time
import datetime
from IPython.display import clear_output

pd.options.plotting.backend = "plotly"

In [30]:
etf_ticker = Ticker("us/etf.yml")
ss_ticker = Ticker("us/equity.yml")

In [34]:
spy_df = etf_ticker.history("SPY", "2022-10-05", "2025-12-12", interval="1d").set_index("ts")

/home/yuqing42/data/stsdata/us/etf/daily/20250908.parquet does not exist, skip


In [35]:
Candle(spy_df).plot()

In [59]:
ticker = Ticker("us/equity.yml")

In [ ]:
df = ticker.history(None, "2025-10-10", "2025-10-10")
df = df.dropna()
df["trdValue"] = df["close"] * df["volume"]
df = df.sort_values("trdValue", ascending=False)
top_500_active_list = df.head(500)["symbol"].to_list()

In [62]:
start = datetime.date(2025, 1, 1)
end = datetime.date(2025, 10, 10)
df = ticker.history(top_500_active_list, start, end, interval="1d")

In [63]:
df.set_index("date", inplace=True)

In [68]:
trend_all_df = {5: [], 20: [], 80: []}
for symbol, gp in df.groupby("symbol"):
    for short in trend_all_df.keys():
        trend = get_trend_score(gp, short=short)
        trend.name = symbol
        trend_all_df[short].append(trend)

In [69]:
for short in trend_all_df.keys():
    trend_all_df[short] = pd.concat(trend_all_df[short], axis=1)

In [97]:
def quantile_mean(x, quantile=[0.1, 0.9]):
    q1, q2 = x.quantile(quantile)
    return x.clip(q1, q2).mean()


mean_df = trend_all_df[5].apply(lambda x: quantile_mean(x, quantile=[0.1, 0.9]), axis=1)

In [108]:
df = pd.read_parquet("~/data/stsdata/us/equity/stats/breath_1d.parquet")

In [110]:
df

,1d_5,1d_20,1d_80
date,,,
2025-10-01,59.6,67.7,59.1
2025-10-02,60.0,67.8,59.1
2025-10-03,61.6,68.3,59.6
2025-10-06,62.1,68.7,59.8
2025-10-07,60.5,67.9,60.1
2025-10-08,60.3,68.2,60.3
2025-10-09,56.9,67.8,60.1
2025-10-10,48.1,66.2,60.0


In [99]:
mean_df.plot()

In [70]:
result = {}
for key in trend_all_df.keys():
    trend_sign = (trend_all_df[key] > 0) * 1
    pos_pct = trend_sign.mean(axis=1) * 100
    result[key] = pos_pct

In [71]:
pd.DataFrame(result).plot()

In [72]:
result = pd.DataFrame(result)

In [81]:
result["20_2"] = result[5].ewm(80).mean()

In [82]:
result.plot()